In [1]:
import os, subprocess

REPO_URL = "https://github.com/tongyuguo/HelpHerInvest.git"
REPO_DIR = "HelpHerInvest"

def clone_or_pull():
    if os.path.isdir(os.path.join(REPO_DIR, ".git")):
        subprocess.run(["git", "-C", REPO_DIR, "pull"])
    else:
        subprocess.run(["git", "clone", REPO_URL])

clone_or_pull()

Already up to date.


In [2]:
import pandas as pd
import zipfile

zip_path = "HelpHerInvest/Data/stock_symbols_new.csv.zip"

with zipfile.ZipFile(zip_path) as z:
    df = pd.read_csv(z.open("stock_symbols_new.csv"))

df.head()

/tmp/ipykernel_2407292/1261020715.py:7: DtypeWarning: Columns (54,204,212,213,214,220,221,222,223) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(z.open("stock_symbols_new.csv"))


,symbol,company_name,address1,city,state,zip,country,phone,website,industry,...,address3,morningStarOverallRating,morningStarRiskRating,annualReportExpenseRatio,lastCapGain,annualHoldingsTurnover,prevTicker,tickerChangeDate,newListingDate,delistingDate
0,NVDA,NVIDIA CORP,2788 San Tomas Expressway,Santa Clara,CA,95051,United States,408 486 2000,https://www.nvidia.com,Semiconductors,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,GOOGL,Alphabet Inc.,1600 Amphitheatre Parkway,Mountain View,CA,94043,United States,650-253-0000,https://abc.xyz,Internet Content & Information,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,AAPL,Apple Inc.,One Apple Park Way,Cupertino,CA,95014,United States,(408) 996-1010,https://www.apple.com,Consumer Electronics,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,MSFT,MICROSOFT CORP,One Microsoft Way,Redmond,WA,98052-6399,United States,425 882 8080,https://www.microsoft.com,Software - Infrastructure,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,AMZN,AMAZON COM INC,410 Terry Avenue North,Seattle,WA,98109-5210,United States,206 266 1000,https://www.amazon.com,Internet Retail,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
#Percent missing per columnn
missing_percent = df.isnull().mean().sort_values(ascending=False) * 100

missing_summary = pd.DataFrame({
    "Missing_Count": df.isnull().sum(),
    "Missing_Percent": missing_percent
}).sort_values(by="Missing_Percent", ascending=False)

missing_summary.head(20)

,Missing_Count,Missing_Percent
annualHoldingsTurnover,10283,99.990276
address3,10283,99.990276
annualReportExpenseRatio,10283,99.990276
delistingDate,10283,99.990276
morningStarRiskRating,10283,99.990276
morningStarOverallRating,10283,99.990276
industrySymbol,10283,99.990276
lastCapGain,10283,99.990276
tickerChangeDate,10283,99.990276
prevTicker,10283,99.990276


In [4]:
#categorize columns by missingness
def categorize_missingness(df):
    missing_ratio = df.isnull().mean()

    categories = {
        "0% Missing": [],
        "1-10% Missing": [],
        "10-50% Missing": [],
        "50-95% Missing": [],
        ">95% Missing": []
    }

    for col, ratio in missing_ratio.items():
        if ratio == 0:
            categories["0% Missing"].append(col)
        elif ratio <= 0.10:
            categories["1-10% Missing"].append(col)
        elif ratio <= 0.50:
            categories["10-50% Missing"].append(col)
        elif ratio <= 0.95:
            categories["50-95% Missing"].append(col)
        else:
            categories[">95% Missing"].append(col)

    return categories

missing_categories = categorize_missingness(df)

for category, cols in missing_categories.items():
    print(f"\n{category}: {len(cols)} columns")


0% Missing: 1 columns

1-10% Missing: 62 columns

10-50% Missing: 87 columns

50-95% Missing: 39 columns

>95% Missing: 35 columns


In [5]:
#categorize columns by missingness
def categorize_missingness(df):
    missing_ratio = df.isnull().mean()

    categories = {
        "0% Missing": [],
        "1-10% Missing": [],
        "10-25% Missing": [],
        "25-50% Missing": [],
        "50-75% Missing": [],
        "75-95% Missing": [],
        ">95% Missing": []
    }

    for col, ratio in missing_ratio.items():
        if ratio == 0:
            categories["0% Missing"].append(col)
        elif ratio <= 0.10:
            categories["1-10% Missing"].append(col)
        elif ratio <= 0.25:
            categories["10-25% Missing"].append(col)
        elif ratio <= 0.50:
            categories["25-50% Missing"].append(col)
        elif ratio <= 0.75:
            categories["50-75% Missing"].append(col)
        elif ratio <= 0.95:
            categories["75-95% Missing"].append(col)
        else:
            categories[">95% Missing"].append(col)

    return categories

missing_categories = categorize_missingness(df)

for category, cols in missing_categories.items():
    print(f"\n{category}: {len(cols)} columns")


0% Missing: 1 columns

1-10% Missing: 62 columns

10-25% Missing: 55 columns

25-50% Missing: 32 columns

50-75% Missing: 33 columns

75-95% Missing: 6 columns

>95% Missing: 35 columns


In [6]:
#Check for duplicate column names 
duplicate_name_mask = df.columns.duplicated()

print("Number of duplicate column names:", duplicate_name_mask.sum())

print("Duplicate column names:")
print(df.columns[duplicate_name_mask])

Number of duplicate column names: 0
Duplicate column names:
Index([], dtype='object')


In [9]:
#Find the columns that are duplicates by identical values
dup_cols = df.T[df.T.duplicated(keep="first")].index.tolist()
print("Duplicate-content columns (these are the 'later' duplicates):")
print(dup_cols)

#For each duplicate column, find the earlier column it matches
matches = {}

for dup in dup_cols:
    for earlier in df.columns:
        if earlier == dup:
            break  
        if df[dup].equals(df[earlier]):
            matches[dup] = earlier
            break

print("\nMapping (duplicate -> identical to):")
for dup, earlier in matches.items():
    print(f"  {dup}  ->  {earlier}")

#Confirm there are 0 mismatched rows for each pair
print("\nSanity check (mismatch counts):")
for dup, earlier in matches.items():
    mismatch = ~((df[dup] == df[earlier]) | (df[dup].isna() & df[earlier].isna()))
    mismatch_count = int(mismatch.sum())
    print(f"  {dup} vs {earlier}: {mismatch_count} mismatched rows")

print("\nPreview (first 5 rows of each duplicate pair):")
for dup, earlier in matches.items():
    print(f"\n--- {dup} (duplicate)  vs  {earlier} (kept) ---")
    display(df[[earlier, dup]].head(5))

Duplicate-content columns (these are the 'later' duplicates):
['sectorDisp', 'symbol.1', 'esgPopulated', 'earningsCallTimestampEnd']

Mapping (duplicate -> identical to):
  sectorDisp  ->  sector
  symbol.1  ->  symbol
  esgPopulated  ->  cryptoTradeable
  earningsCallTimestampEnd  ->  earningsCallTimestampStart

Sanity check (mismatch counts):
  sectorDisp vs sector: 0 mismatched rows
  symbol.1 vs symbol: 0 mismatched rows
  esgPopulated vs cryptoTradeable: 0 mismatched rows
  earningsCallTimestampEnd vs earningsCallTimestampStart: 0 mismatched rows

Preview (first 5 rows of each duplicate pair):

--- sectorDisp (duplicate)  vs  sector (kept) ---


,sector,sectorDisp
0,Technology,Technology
1,Communication Services,Communication Services
2,Technology,Technology
3,Technology,Technology
4,Consumer Cyclical,Consumer Cyclical



--- symbol.1 (duplicate)  vs  symbol (kept) ---


,symbol,symbol.1
0,NVDA,NVDA
1,GOOGL,GOOGL
2,AAPL,AAPL
3,MSFT,MSFT
4,AMZN,AMZN



--- esgPopulated (duplicate)  vs  cryptoTradeable (kept) ---


,cryptoTradeable,esgPopulated
0,False,False
1,False,False
2,False,False
3,False,False
4,False,False



--- earningsCallTimestampEnd (duplicate)  vs  earningsCallTimestampStart (kept) ---


,earningsCallTimestampStart,earningsCallTimestampEnd
0,1.763590e+09,1.763590e+09
1,1.770241e+09,1.770241e+09
2,1.769724e+09,1.769724e+09
3,1.769639e+09,1.769639e+09
4,1.761858e+09,1.761858e+09


In [ ]:
#Drop the confirmed duplicate-content columns
cols_to_drop = ["sectorDisp", "symbol.1", "esgPopulated", "earningsCallTimestampEnd"]
df = df.drop(columns=cols_to_drop)

print("New shape after dropping duplicates:", df.shape)

#Save cleaned version
df.to_csv("stock_symbols_new_dropped_columns.csv", index=False)
print("Saved: stock_symbols_new_dropped_columns.csv")

In [3]:
def can_push():
    result = subprocess.run(
        ["git", "-C", REPO_DIR, "push", "--dry-run"],
        capture_output=True, text=True
    )
    return result.returncode == 0

def commit_and_push(msg="update"):
    subprocess.run(["git", "-C", REPO_DIR, "add", "-A"])
    subprocess.run(["git", "-C", REPO_DIR, "commit", "-m", msg])
    subprocess.run(["git", "-C", REPO_DIR, "push"])

if can_push():
    commit_and_push("update notebook / data")
else:
    print("Not a collaborator — skipping push")

Not a collaborator — skipping push
